# Pronóstico con regresores externos usando Prophet

## Introducción del ejercicio

Una serie temporal puede depender de factores externos además de su propia historia. Por ejemplo, la demanda de una tienda digital puede aumentar cuando se invierte más en publicidad o cuando se realiza una campaña especial.

El problema que resolveremos es: **¿cuántas órdenes diarias podemos esperar considerando tanto el comportamiento histórico como el presupuesto de publicidad y los eventos especiales planeados?**

Prophet permite agregar estas variables mediante `add_regressor`. En este ejercicio compararemos un modelo que solo conoce la historia de las órdenes contra otro que también recibe inversión publicitaria y eventos especiales.

## Objetivos

Al finalizar podremos:

1. Construir un dataset reproducible con variables externas.
2. Distinguir la variable objetivo de los regresores.
3. Entrenar Prophet con y sin regresores.
4. Evaluar si los regresores mejoran el pronóstico.
5. Generar un pronóstico futuro usando un plan conocido de publicidad y eventos.
6. Comprender las limitaciones y supuestos de este enfoque.

## 1. Preparar el entorno

Usaremos Prophet para el modelo, `pandas` para preparar la información, `numpy` para crear un ejemplo reproducible, `matplotlib` para las gráficas y `scikit-learn` para las métricas.

In [ ]:
%pip install -q prophet scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

## 2. Cargar el dataset de demanda y regresores

El archivo `prophet_regresores_demanda.csv` contiene 270 días de órdenes. La demanda incluye una tendencia suave, un patrón semanal, inversión en publicidad y eventos especiales.

- `y`: órdenes diarias, la variable que queremos pronosticar.
- `marketing_spend`: inversión diaria en publicidad.
- `evento_especial`: indicador que vale 1 cuando existe una campaña o evento.

El archivo debe contener las columnas `ds`, `y`, `marketing_spend` y `evento_especial`.

In [ ]:
from google.colab import files

archivos_subidos = files.upload()
nombre_archivo = next(iter(archivos_subidos))
datos = pd.read_csv(nombre_archivo)
datos['ds'] = pd.to_datetime(datos['ds'])

columnas_requeridas = {'ds', 'y', 'marketing_spend', 'evento_especial'}
if not columnas_requeridas.issubset(datos.columns):
    faltantes = columnas_requeridas - set(datos.columns)
    raise ValueError(f'Faltan columnas requeridas: {faltantes}')

datos = datos.sort_values('ds').reset_index(drop=True)
print(f'Archivo cargado: {nombre_archivo}')
print(f'Registros: {len(datos):,}')
datos.head()

## 3. Explorar la variable objetivo y los regresores

Antes de entrenar, observaremos la relación temporal entre órdenes, publicidad y eventos. La publicidad puede variar día a día y los eventos producen incrementos puntuales. Prophet no debe recibir estas variables solo como información histórica: también necesitaremos sus valores para las fechas futuras.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
datos.plot(x='ds', y='y', ax=axes[0], color='#2563eb', legend=False)
axes[0].set_title('Órdenes diarias')
axes[0].set_ylabel('Órdenes')
datos.plot(x='ds', y='marketing_spend', ax=axes[1], color='#f59e0b', legend=False)
axes[1].set_title('Inversión diaria en publicidad')
axes[1].set_ylabel('Presupuesto')
fechas_numericas = mdates.date2num(datos['ds'].tolist())
axes[1].fill_between(fechas_numericas, 0, datos['evento_especial'].to_numpy() * datos['marketing_spend'].max(), color='#dc2626', alpha=0.2, label='Evento especial')
axes[2].plot(datos['ds'], datos['evento_especial'], color='#dc2626')
axes[2].set_title('Indicador de evento especial')
axes[2].set_ylabel('Evento')
axes[2].set_xlabel('Fecha')
plt.tight_layout()

## 4. Separar entrenamiento y prueba

Reservaremos los últimos 30 días como prueba. La variable objetivo de esas fechas se ocultará durante el entrenamiento, pero los regresores sí estarán disponibles porque representan un plan de publicidad y eventos que se conoce por anticipado. Esta es una condición fundamental para utilizar regresores externos en un pronóstico real.

In [ ]:
horizonte_prueba = 30
entrenamiento = datos.iloc[:-horizonte_prueba].copy()
prueba = datos.iloc[-horizonte_prueba:].copy()

print(f'Entrenamiento: {entrenamiento.ds.min():%Y-%m-%d} a {entrenamiento.ds.max():%Y-%m-%d}')
print(f'Prueba: {prueba.ds.min():%Y-%m-%d} a {prueba.ds.max():%Y-%m-%d}')
print('Regresores disponibles en prueba:', ['marketing_spend', 'evento_especial'])

## 5. Crear un modelo Prophet sin regresores

Primero construiremos una referencia que usa únicamente la historia de las órdenes y la estacionalidad semanal. Este modelo base permite medir si agregar publicidad y eventos realmente aporta información adicional.

In [ ]:
modelo_base = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95
).fit(entrenamiento[['ds', 'y']])

pred_base = modelo_base.predict(prueba[['ds']])
print('Modelo base entrenado.')

## 6. Crear Prophet con regresores externos

Ahora agregaremos las dos variables externas con `add_regressor`. Prophet estimará cuánto cambia el nivel esperado de órdenes cuando cambia la inversión publicitaria y cuando ocurre un evento especial. `standardize=True` es el comportamiento predeterminado y ayuda a que los regresores estén en escalas comparables.

In [ ]:
modelo_regresores = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95
)
modelo_regresores.add_regressor('marketing_spend')
modelo_regresores.add_regressor('evento_especial')
modelo_regresores.fit(entrenamiento[['ds', 'y', 'marketing_spend', 'evento_especial']])

pred_regresores = modelo_regresores.predict(
    prueba[['ds', 'marketing_spend', 'evento_especial']]
)
print('Modelo con regresores externos entrenado.')

## 7. Comparar las métricas

Compararemos ambos modelos sobre exactamente las mismas fechas. Si el modelo con regresores obtiene menor MAE, RMSE y MAPE, significa que las variables externas ayudaron a explicar parte de la variación de la demanda.

In [ ]:
def calcular_metricas(real, predicho):
    real = np.asarray(real, dtype=float)
    predicho = np.asarray(predicho, dtype=float)
    mascara_valida = np.isfinite(real) & np.isfinite(predicho) & (real != 0)
    if not mascara_valida.any():
        mape = np.nan
    else:
        mape = np.mean(np.abs((real[mascara_valida] - predicho[mascara_valida]) / real[mascara_valida])) * 100
    return pd.Series({
        'MAE': mean_absolute_error(real, predicho),
        'RMSE': np.sqrt(mean_squared_error(real, predicho)),
        'MAPE (%)': mape
    })

metricas = pd.DataFrame({
    'Prophet base': calcular_metricas(prueba['y'], pred_base['yhat']),
    'Prophet con regresores': calcular_metricas(prueba['y'], pred_regresores['yhat'])
}).T

metricas.round(2)

## 8. Interpretar la comparación

La mejora de métricas debe acompañarse de una interpretación del negocio. En este dataset, la inversión publicitaria y los eventos fueron incluidos como causas conocidas de aumentos en las órdenes. Si el modelo con regresores mejora, demuestra que Prophet puede incorporar información externa para explicar variaciones que la historia de la demanda por sí sola no captura.

In [ ]:
ceros_real = int((prueba['y'] == 0).sum())
mape_base = metricas.loc['Prophet base', 'MAPE (%)']
mape_regresores = metricas.loc['Prophet con regresores', 'MAPE (%)']
mejora = (1 - mape_regresores / mape_base) * 100

print(f'Valores reales iguales a cero excluidos del MAPE: {ceros_real}')
print(f'MAPE del modelo base: {mape_base:.2f}%')
print(f'MAPE con regresores: {mape_regresores:.2f}%')
if mejora > 0:
    print(f'Los regresores reducen el MAPE en {mejora:.2f}%.')
else:
    print(f'Los regresores no mejoran el MAPE en este periodo: cambio de {mejora:.2f}%.')
print('La mejora debe confirmarse con varios periodos de validación y no solo con una partición.')

## 9. Visualizar los pronósticos

La gráfica muestra si el modelo con regresores sigue mejor los picos asociados con campañas y eventos. También permite observar que el modelo base puede producir una trayectoria razonable, pero perder cambios puntuales que son explicados por la información externa.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(entrenamiento['ds'].tail(90), entrenamiento['y'].tail(90), label='Entrenamiento reciente', color='#64748b')
plt.plot(prueba['ds'], prueba['y'], label='Real', color='#111827', linewidth=2)
plt.plot(prueba['ds'], pred_base['yhat'], '--', label='Prophet base', color='#ef4444')
plt.plot(prueba['ds'], pred_regresores['yhat'], '--', label='Prophet con regresores', color='#16a34a', linewidth=2)
plt.scatter(prueba.loc[prueba['evento_especial'] == 1, 'ds'], prueba.loc[prueba['evento_especial'] == 1, 'y'], color='#7c3aed', s=60, label='Evento especial', zorder=5)
plt.axvline(prueba['ds'].iloc[0], color='black', linestyle=':', label='Inicio de prueba')
plt.title('Comparación de Prophet con y sin regresores externos')
plt.xlabel('Fecha')
plt.ylabel('Órdenes')
plt.legend()
plt.tight_layout()

## 10. Revisar el efecto estimado de los regresores

Prophet guarda los coeficientes estimados de los regresores. Debido a que las variables pueden estandarizarse internamente, estos coeficientes sirven principalmente para analizar dirección y relevancia relativa; no deben interpretarse automáticamente como una relación causal definitiva.

In [ ]:
coeficientes = modelo_regresores.params['beta'][0]
nombres_regresores = list(modelo_regresores.extra_regressors.keys())
pd.Series(coeficientes[-len(nombres_regresores):], index=nombres_regresores, name='Coeficiente estimado')

## 11. Pronosticar los próximos 30 días

Para generar un pronóstico futuro necesitamos conocer o construir un plan de publicidad y eventos para esos días. En un caso real, `marketing_spend_futuro` provendría del presupuesto aprobado y `evento_especial_futuro` del calendario comercial. Aquí crearemos un plan ilustrativo con campañas al inicio de cada periodo de 30 días.

In [ ]:
modelo_final = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95
)
modelo_final.add_regressor('marketing_spend')
modelo_final.add_regressor('evento_especial')
modelo_final.fit(datos[['ds', 'y', 'marketing_spend', 'evento_especial']])

fechas_futuras = modelo_final.make_future_dataframe(periods=30, freq='D', include_history=False)
indice_futuro = np.arange(len(fechas_futuras))
marketing_spend_futuro = 80 + 25 * (indice_futuro % 30 == 0)
evento_especial_futuro = ((indice_futuro % 30 == 14) | (indice_futuro % 30 == 15)).astype(int)

futuro_con_regresores = fechas_futuras.assign(
    marketing_spend=marketing_spend_futuro,
    evento_especial=evento_especial_futuro
)
pronostico_futuro = modelo_final.predict(futuro_con_regresores)

tabla_futuro = pronostico_futuro[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
tabla_futuro.columns = ['Fecha', 'Pronóstico', 'Límite inferior 95%', 'Límite superior 95%']
tabla_futuro.head(10).round(2)

## Interpretación del pronóstico futuro

El pronóstico futuro depende del plan de regresores. Si se cambia el presupuesto de publicidad o se agregan eventos especiales, también cambiará el pronóstico. Esta es una ventaja para la planeación de escenarios, pero también una responsabilidad: el modelo no puede saber automáticamente cuánto se invertirá en el futuro.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(datos['ds'].tail(90), datos['y'].tail(90), label='Histórico reciente', color='#2563eb')
plt.plot(pronostico_futuro['ds'], pronostico_futuro['yhat'], label='Pronóstico próximos 30 días', color='#16a34a', linewidth=2)
fechas_futuras_numericas = mdates.date2num(pronostico_futuro['ds'].tolist())
plt.fill_between(fechas_futuras_numericas, pronostico_futuro['yhat_lower'].to_numpy(), pronostico_futuro['yhat_upper'].to_numpy(), color='#86efac', alpha=0.3, label='Intervalo 95%')
plt.scatter(futuro_con_regresores.loc[futuro_con_regresores['evento_especial'] == 1, 'ds'], pronostico_futuro.loc[futuro_con_regresores['evento_especial'] == 1, 'yhat'], color='#7c3aed', s=60, label='Evento futuro', zorder=5)
plt.axvline(datos['ds'].max(), color='black', linestyle=':', label='Último dato disponible')
plt.title('Pronóstico futuro condicionado por publicidad y eventos')
plt.xlabel('Fecha')
plt.ylabel('Órdenes')
plt.legend()
plt.tight_layout()

## Conclusiones generales

- Los regresores externos permiten que Prophet utilice información adicional a la historia de la variable objetivo.
- En este caso, publicidad y eventos ayudan a explicar aumentos de órdenes que un modelo basado solo en estacionalidad podría perder.
- La comparación contra un modelo base es necesaria para comprobar que los regresores agregan valor.
- Para pronosticar el futuro, los valores de los regresores deben ser conocidos, presupuestados o estimados mediante escenarios.
- Una mejora de precisión no demuestra por sí misma causalidad; se requiere diseño experimental o análisis adicional para afirmar efectos causales.
- En un proyecto real convendría validar en varios periodos, revisar correlaciones, evitar regresores que solo estén disponibles después del resultado y monitorear cambios en la relación entre publicidad y demanda.